## Custom Autoscaling in Ray Serve

This notebook explores Ray Serve's custom autoscaling capabilities, which allow you to implement domain-specific scaling logic beyond the default queue-depth policy.

<div class="alert alert-info">
<b>Here is the roadmap of this notebook:</b>
<ol>
  <li>Overview of Custom Autoscaling</li>
  <li>Deployment-Level Policies
    <ul>
      <li>Schedule-Based Autoscaling (Time of Day)</li>
      <li>Recorded Stats-Based Autoscaling (CPU/Memory)</li>
      <li>Prometheus-Based Autoscaling (Latency SLA)</li>
    </ul>
  </li>
  <li>Application-Level Policies (Multi-Deployment Coordination)</li>
  <li>Best Practices</li>
  <li>Connecting an External Autoscaling Service</li>
</ol>
</div>

**imports**

In [ ]:
import asyncio
import logging
import os
from datetime import datetime
from zoneinfo import ZoneInfo
from typing import Any, Dict, Tuple

from ray import serve
from ray.serve.config import AutoscalingConfig, AutoscalingPolicy, AutoscalingContext
from ray.serve._private.common import DeploymentID

## 1. Overview of Custom Autoscaling

Ray Serve's default autoscaling uses a **queue-depth policy** that scales based on `target_ongoing_requests`. While this works well for many workloads, some scenarios require more sophisticated scaling logic:

| Use Case | Why Custom Policy? |
|----------|--------------------|
| **Predictable traffic** | Pre-scale based on time of day or scheduled events |
| **Resource-aware** | Scale based on CPU/memory/GPU utilization |
| **Latency SLA** | Scale based on P90/P99 latency rather than queue depth |
| **Multi-deployment pipelines** | Coordinate scaling across dependent services |

### How Custom Policies Work

Custom policies are Python functions that:
1. Receive an `AutoscalingContext` with current state and metrics
2. Implement custom logic to determine the target replica count
3. Return the target replica count and optional state to persist

```python
def my_autoscaling_policy(
    ctx: AutoscalingContext,
) -> tuple[int, Dict[str, Any]]:
    # Your custom logic here
    target_replicas = ...
    policy_state = ...
    return target_replicas, policy_state
```

### Key Concepts in `AutoscalingContext`

The `AutoscalingContext` provides everything your policy needs:

| Attribute | Description |
|-----------|-------------|
| `target_num_replicas` | **Use this as baseline** - accounts for pending scaling decisions |
| `current_num_replicas` | Actual running replicas (may lag behind target) |
| `capacity_adjusted_min_replicas` | Minimum replicas (respect this bound) |
| `capacity_adjusted_max_replicas` | Maximum replicas (respect this bound) |
| `total_num_requests` | Total queued + ongoing requests across all replicas |
| `aggregated_metrics` | Custom metrics from `record_autoscaling_stats()` |
| `policy_state` | Persisted state from previous policy invocation |
| `config` | The `AutoscalingConfig` for this deployment |

## 2. Deployment-Level Policies

Deployment-level policies make scaling decisions for a single deployment. These are the most common type of custom policy.

### 2.1 Schedule-Based Autoscaling (Time of Day)

This custom policy type scales based on **time of day**, useful for applications with predictable traffic patterns.

**Use Case**: Your application receives heavy traffic during business hours (9am-5pm) but minimal traffic overnight. Pre-scale to avoid cold-start latency during peak times.

#### The Policy Function

```python
# From: scripts/custom-autoscaling-experiments-main/deployment_level/schedule_based/autoscaling_policy.py
```

This policy simply checks the current hour and returns the appropriate replica count. No external dependencies, no state management - just straightforward logic:

In [ ]:
def scheduled_autoscaling_policy(
    ctx: AutoscalingContext,
) -> tuple[int, Dict[str, Any]]:
    """Scale based on time of day."""
    current_hour = datetime.now(ZoneInfo("America/Los_Angeles")).hour

    # Define replica targets by hour (customize for your traffic pattern)
    if 9 <= current_hour < 17:  # Business hours: 9am-5pm
        desired = 8
    elif 7 <= current_hour < 9 or 17 <= current_hour < 20:  # Shoulder hours
        desired = 4
    else:  # Off hours
        desired = 1

    # Respect configured bounds
    final = max(
        ctx.capacity_adjusted_min_replicas,
        min(ctx.capacity_adjusted_max_replicas, desired),
    )

    return final, {"reason": f"Hour {current_hour}: target={desired}"}

#### Deploying with Schedule-Based Policy

To use a custom policy, configure the deployment with `AutoscalingPolicy` pointing to your policy function:

In [ ]:
@serve.deployment(
    autoscaling_config=AutoscalingConfig(
        min_replicas=1,
        max_replicas=12,
        upscale_delay_s=30,
        downscale_delay_s=60,
        policy=AutoscalingPolicy(
            # When running from file, use: "autoscaling_policy:scheduled_autoscaling_policy"
            policy_function=scheduled_autoscaling_policy
        ),
    ),
    max_ongoing_requests=5,
)
class ScheduledDeployment:
    async def __call__(self) -> str:
        await asyncio.sleep(0.5)  # Simulate processing
        return "Hello, world!"

#### Key Points

- **Proactive Scaling**: Replicas are ready before traffic arrives
- **Cost Control**: Automatically scales down during off-peak hours
- **Combine with Reactive**: Consider combining with queue-depth metrics for spike handling
- **Timezone Awareness**: Consider using `datetime.now(timezone)` for multi-region deployments

In [ ]:
!cd examples/custom-autoscaling-experiments/deployment_level/schedule_based/ && serve run main:app

We can see the autoscaling policy in action in the logs below (Running between 7am and 9am Pacific Time):

```text
Deploying new version of Deployment(name='ProcessingDeployment', app='default') (initial target replicas: 1).
Adding 1 replica to Deployment(name='ProcessingDeployment', app='default').
Starting Replica(id='8qpjdgl5', deployment='ProcessingDeployment', app='default').
Adding 3 replicas to Deployment(name='ProcessingDeployment', app='default').
Starting Replica(id='50rn4s4q', deployment='ProcessingDeployment', app='default').
Starting Replica(id='6oyvrne2', deployment='ProcessingDeployment', app='default').
Starting Replica(id='evwnyxnp', deployment='ProcessingDeployment', app='default').
```


### 2.2 Recorded Stats-Based Autoscaling (CPU/Memory)

This policy adds **custom metrics collection** to the mix. It scales based on **CPU and memory usage** reported directly by replicas via the `record_autoscaling_stats()` method. No external monitoring system required!

**Use Case**: Scale based on actual resource consumption rather than request count. Useful for workloads where processing cost varies significantly per request.

#### The Policy Function

```python
# From: scripts/custom-autoscaling-experiments-main/deployment_level/recorded_stats_based/autoscaling_policy.py
```

The policy:
1. Reads CPU and memory metrics from `ctx.aggregated_metrics`
2. Scales up if **any** replica is overloaded (max usage > threshold)
3. Scales down if **all** replicas are underutilized (avg usage < threshold)

In [6]:
# Resource thresholds (customize based on your workload)
CPU_HIGH = 75.0  # Scale up when any replica exceeds this
CPU_LOW = 25.0   # Scale down when all replicas below this
MEMORY_HIGH_MB = 500.0
MEMORY_LOW_MB = 200.0


def custom_metrics_autoscaling_policy(
    ctx: AutoscalingContext,
) -> tuple[int, Dict[str, Any]]:
    """Scale based on CPU and memory metrics from replicas."""
    cpu_metrics = ctx.aggregated_metrics.get("cpu_usage", {})
    memory_metrics = ctx.aggregated_metrics.get("memory_usage_mb", {})

    if not cpu_metrics or not memory_metrics:
        return ctx.target_num_replicas, {"reason": "No metrics available"}

    cpu_values = list(cpu_metrics.values())
    memory_values = list(memory_metrics.values())

    max_cpu = max(cpu_values)
    max_memory = max(memory_values)
    avg_cpu = sum(cpu_values) / len(cpu_values)
    avg_memory = sum(memory_values) / len(memory_values)

    # Scale up if any replica is overloaded
    if max_cpu > CPU_HIGH or max_memory > MEMORY_HIGH_MB:
        new_replicas = min(
            ctx.capacity_adjusted_max_replicas,
            ctx.target_num_replicas + 1,
        )
        return new_replicas, {
            "reason": f"High usage: CPU={max_cpu:.0f}%, Mem={max_memory:.0f}MB"
        }

    # Scale down if all replicas are underutilized
    if avg_cpu < CPU_LOW and avg_memory < MEMORY_LOW_MB:
        new_replicas = max(
            ctx.capacity_adjusted_min_replicas,
            ctx.target_num_replicas - 1,
        )
        return new_replicas, {
            "reason": f"Low usage: CPU={avg_cpu:.0f}%, Mem={avg_memory:.0f}MB"
        }

    return ctx.target_num_replicas, {
        "reason": f"Stable: CPU={avg_cpu:.0f}%, Mem={avg_memory:.0f}MB"
    }

#### Deploying with Custom Metrics

The key is implementing `record_autoscaling_stats()` in your deployment to report metrics:

In [7]:
import psutil

@serve.deployment(
    autoscaling_config=AutoscalingConfig(
        min_replicas=1,
        max_replicas=10,
        upscale_delay_s=30,
        downscale_delay_s=60,
        policy=AutoscalingPolicy(
            policy_function=custom_metrics_autoscaling_policy
        ),
    ),
    max_ongoing_requests=5,
)
class CustomMetricsDeployment:
    def __init__(self):
        self.process = psutil.Process()

    async def __call__(self) -> str:
        await asyncio.sleep(0.5)  # Simulate processing
        return "Hello, world!"

    def record_autoscaling_stats(self) -> Dict[str, float]:
        """Report custom metrics to the autoscaler.
        
        This method is called periodically by Ray Serve. The returned metrics
        are available in the policy via ctx.aggregated_metrics.
        """
        try:
            return {
                "cpu_usage": self.process.cpu_percent(interval=0.1),
                "memory_usage_mb": self.process.memory_info().rss / (1024 * 1024),
            }
        except Exception:
            return {"cpu_usage": 0.0, "memory_usage_mb": 0.0}

#### Key Points

- **`record_autoscaling_stats()`**: Must be a method on your deployment class that returns a `Dict[str, float]`
- **Called Periodically**: Ray Serve calls this method on each replica and aggregates the results
- **Metric Aggregation**: `ctx.aggregated_metrics["cpu_usage"]` is a dict mapping replica ID to the reported value
- **No External Dependencies**: Unlike Prometheus-based, this works out of the box

### 2.3 Prometheus-Based Autoscaling (Latency SLA)

This is the most **advanced** deployment-level policy. It scales based on **P90 latency metrics** fetched from Prometheus, and includes manual delay logic using `policy_state`.

**Use Case**: You have an SLA that P90 latency must stay under 500ms. Scale up when latency exceeds threshold, scale down when latency is well below threshold.

#### The Policy Function

```python
# From: scripts/custom-autoscaling-experiments-main/deployment_level/prometheus_based/autoscaling_policy.py
```

The policy:
1. Queries Prometheus for P90 latency using Ray Serve's built-in metrics
2. Compares against configurable thresholds (`SCALE_UP_THRESHOLD_MS`, `SCALE_DOWN_THRESHOLD_MS`)
3. Implements delay logic to avoid thrashing (sustained signal before scaling)

In [ ]:
import requests
import ray
from ray.serve._private.constants import CONTROL_LOOP_INTERVAL_S

logger = logging.getLogger("ray.serve")

PROMETHEUS_HOST = os.environ.get("RAY_PROMETHEUS_HOST", "http://localhost:9090")

# Latency thresholds (customize based on your SLA)
SCALE_UP_THRESHOLD_MS = 500
SCALE_DOWN_THRESHOLD_MS = 50


def get_p90_latency(app_name: str, deployment_name: str) -> float | None:
    """Fetch P90 latency from Prometheus."""
    session_name = ray._private.worker.global_worker.node.session_name
    query = (
        "histogram_quantile(0.9, "
        "sum(rate(ray_serve_deployment_processing_latency_ms_bucket{"
        f'application="{app_name}",'
        f'deployment="{deployment_name}",'
        f'SessionName="{session_name}"'
        "}[5m])) by (application, deployment, le))"
    )
    try:
        response = requests.get(
            f"{PROMETHEUS_HOST}/api/v1/query",
            params={"query": query},
            timeout=5,
        )
        response.raise_for_status()
        result = response.json()
        if result["status"] == "success" and result["data"]["result"]:
            return float(result["data"]["result"][0]["value"][1])
    except Exception as e:
        logger.warning(f"Failed to query Prometheus: {e}")
    return None


def p90_latency_autoscaling_policy(
    ctx: AutoscalingContext,
) -> tuple[int, Dict[str, Any]]:
    """Scale based on P90 latency from Prometheus."""
    policy_state = ctx.policy_state or {}
    decision_counter = policy_state.get("decision_counter", 0)

    p90_latency_ms = get_p90_latency(ctx.app_name, ctx.deployment_name)

    if p90_latency_ms is None:
        return ctx.target_num_replicas, policy_state

    # Determine scaling direction based on latency
    if p90_latency_ms > SCALE_UP_THRESHOLD_MS:
        # Want to scale up
        desired = min(ctx.capacity_adjusted_max_replicas, ctx.target_num_replicas + 1)
        if desired > ctx.target_num_replicas:
            decision_counter = max(1, decision_counter + 1)
        else:
            decision_counter = 0

    elif p90_latency_ms < SCALE_DOWN_THRESHOLD_MS:
        # Want to scale down
        desired = max(ctx.capacity_adjusted_min_replicas, ctx.target_num_replicas - 1)
        if desired < ctx.target_num_replicas:
            decision_counter = min(-1, decision_counter - 1)
        else:
            decision_counter = 0
    else:
        # Latency acceptable - reset counter
        decision_counter = 0
        desired = ctx.target_num_replicas

    # Apply delay logic: only scale after sustained signal
    final_replicas = ctx.target_num_replicas
    upscale_threshold = int(ctx.config.upscale_delay_s / CONTROL_LOOP_INTERVAL_S)
    downscale_threshold = int(ctx.config.downscale_delay_s / CONTROL_LOOP_INTERVAL_S)

    if decision_counter > upscale_threshold:
        final_replicas = desired
        decision_counter = 0
    elif decision_counter < -downscale_threshold:
        final_replicas = desired
        decision_counter = 0

    policy_state["decision_counter"] = decision_counter
    return final_replicas, policy_state

#### Deploying with Prometheus-Based Policy

In [ ]:
@serve.deployment(
    autoscaling_config=AutoscalingConfig(
        min_replicas=1,
        max_replicas=12,
        upscale_delay_s=30,
        downscale_delay_s=60,
        policy=AutoscalingPolicy(
            policy_function=p90_latency_autoscaling_policy
        ),
    ),
    max_ongoing_requests=5,
)
class LatencySLADeployment:
    async def __call__(self) -> str:
        await asyncio.sleep(0.5)  # Simulate processing
        return "Hello, world!"

#### Key Points

- **Prometheus Required**: Ensure `RAY_PROMETHEUS_HOST` environment variable is set
- **Manual Delay Logic**: The policy manually implements `upscale_delay_s` and `downscale_delay_s` using `decision_counter` and `policy_state`
- **Graceful Fallback**: Returns current replica count if Prometheus is unavailable

## 3. Application-Level Policies (Multi-Deployment Coordination)

Application-level policies coordinate scaling decisions across **multiple deployments** in a single application. This is essential for pipeline architectures where deployments have dependencies.

**Use Case**: A preprocessing -> model pipeline where the model takes 2x longer than preprocessing. Scale the model deployment at 2x the rate of the preprocessor to prevent bottlenecks.

#### The Policy Function

```python
# From: scripts/custom-autoscaling-experiments-main/application_level/autoscaling_policy.py
```

Application-level policies receive a **dictionary** of contexts (one per deployment) and return scaling decisions for all deployments:

In [ ]:
def coordinated_scaling_policy(
    contexts: Dict[DeploymentID, AutoscalingContext],
) -> Tuple[Dict[DeploymentID, int], Dict]:
    """Scale deployments proportionally based on a preprocessing -> model pipeline."""
    decisions = {}

    # Find deployments by name (safely)
    preprocessing_id = next((d for d in contexts if d.name == "Preprocessor"), None)
    model_id = next((d for d in contexts if d.name == "Model"), None)

    if preprocessing_id is None or model_id is None:
        # Return empty decisions if deployments not found
        return {}, {"error": "Required deployments not found"}

    preprocessing_ctx = contexts[preprocessing_id]
    model_ctx = contexts[model_id]

    # Scale preprocessor: 1 replica per 10 queued requests
    desired = max(1, int(preprocessing_ctx.total_num_requests // 10))
    # Apply user-defined bounds of min and max replicas
    preprocessing_replicas = min(desired, preprocessing_ctx.capacity_adjusted_max_replicas)
    preprocessing_replicas = max(preprocessing_replicas, preprocessing_ctx.capacity_adjusted_min_replicas)
    decisions[preprocessing_id] = preprocessing_replicas

    # Scale model: 2x preprocessor (model takes longer)
    desired = preprocessing_replicas * 2
    # Apply user-defined bounds of min and max replicas
    model_replicas = min(desired, model_ctx.capacity_adjusted_max_replicas)
    model_replicas = max(model_replicas, model_ctx.capacity_adjusted_min_replicas)
    decisions[model_id] = model_replicas

    return decisions, {}

#### Function Signature Differences

| Level | Input | Output |
|-------|-------|--------|
| **Deployment** | `AutoscalingContext` | `tuple[int, Dict]` |
| **Application** | `Dict[DeploymentID, AutoscalingContext]` | `tuple[Dict[DeploymentID, int], Dict]` |

#### Key Points

- **Joint Decisions**: Make coordinated scaling decisions based on the full application state
- **Maintain Ratios**: Useful for maintaining replica ratios (e.g., 2:1 model:preprocessor)
- **Pipeline Awareness**: Prevents bottlenecks by scaling downstream services proportionally
- **Graceful Handling**: Return empty decisions if expected deployments aren't found

## 4. Best Practices

When implementing custom autoscaling policies, follow these guidelines:

### Always Use `target_num_replicas` as Baseline

```python
# Good: Use target_num_replicas
new_replicas = ctx.target_num_replicas + 1

# Bad: Don't use current_num_replicas
new_replicas = ctx.current_num_replicas + 1  # May lag behind
```

The `target_num_replicas` reflects pending scaling decisions, while `current_num_replicas` may lag.

### Always Respect Bounds

```python
# Good: Clamp to bounds
final = max(
    ctx.capacity_adjusted_min_replicas,
    min(ctx.capacity_adjusted_max_replicas, desired),
)

# Bad: Ignoring bounds
final = desired  # May exceed max or go below min
```

### Use `policy_state` for Delay Logic

```python
def my_policy(ctx: AutoscalingContext) -> tuple[int, Dict[str, Any]]:
    policy_state = ctx.policy_state or {}
    counter = policy_state.get("decision_counter", 0)
    
    # ... scaling logic that updates counter ...
    
    policy_state["decision_counter"] = counter
    return final_replicas, policy_state  # State persisted for next call
```

## Running the Examples

The example scripts can be run directly with `serve run`:

```bash
# Schedule-based (simplest - no dependencies)
serve run scripts/custom-autoscaling-experiments-main/deployment_level/schedule_based/main:app

# Recorded stats-based
serve run scripts/custom-autoscaling-experiments-main/deployment_level/recorded_stats_based/main:app

# Prometheus-based (requires Prometheus running)
export RAY_PROMETHEUS_HOST=http://localhost:9090
serve run scripts/custom-autoscaling-experiments-main/deployment_level/prometheus_based/main:app
```

## 5. Connecting an External Autoscaling Service

The external scaling API provides programmatic control over the number of replicas for any deployment. Unlike Ray Serve's built-in autoscaling (which scales based on queue depth and ongoing requests), this API allows you to scale based on any external criteria—time of day, predictive models, external metrics, or custom business logic.

**Quick Reference:**
- **Scale endpoint:** `POST /api/v1/applications/{app}/deployments/{dep}/scale` with `{"target_num_replicas": N}`
- **Query status:** `GET /api/serve/applications/`
- **Enable:** Set `external_scaler_enabled: true` in your application config

> **Warning:** External scaling and built-in autoscaling are mutually exclusive. If you set `external_scaler_enabled: true`, do not configure `autoscaling_config` on any deployment in that application.

### Example: Predictive Scaling

This example shows how to implement predictive scaling based on time of day. You can preemptively scale up before anticipated traffic spikes by running an external script that adjusts replica counts.

```python
# From: examples/custom-autoscaling-experiments/external_scaler/external_scaler_predictive.py
```

**Step 1: Define the deployment**

The deployment is a simple text processor that can be scaled externally:

```python
@serve.deployment(num_replicas=3)
class TextProcessor:
    """A simple text processing deployment that can be scaled externally."""
    def __call__(self, text: Any) -> dict:
        time.sleep(0.1)  # Simulate processing
        return {"request_count": self.request_count}

app = TextProcessor.bind()
```

**Step 2: Configure external scaling**

Enable external scaling in your application config (`external_scaler_config.yaml`):

```yaml
applications:
  - name: my-app
    import_path: external_scaler_predictive:app
    external_scaler_enabled: true  # Key setting!
    deployments:
      - name: TextProcessor
        num_replicas: 1
```

**Step 3: Implement the scaling logic**

```python
# From: examples/custom-autoscaling-experiments/external_scaler/external_scaler_predictive_client.py
```

The scaling client queries the current replica count and adjusts based on time of day:

```python
def scale_deployment(app_name: str, deployment_name: str):
    """Scale deployment based on time of day."""
    hour = datetime.now().hour
    current = get_current_replicas(app_name, deployment_name)
    
    target = 10 if 9 <= hour < 17 else 3  # Peak hours: 9am-5pm
    
    # Call the external scaling API
    resp = requests.post(
        f"{SERVE_ENDPOINT}/api/v1/applications/{app_name}/deployments/{deployment_name}/scale",
        json={"target_num_replicas": target},
    )
```

The client continuously adjusts replicas:
- **Business hours (9 AM - 5 PM):** 10 replicas
- **Off-peak hours:** 3 replicas

**Step 4: Run the example**

Start the Ray Serve application:

```bash
cd examples/custom-autoscaling-experiments/external_scaler/
serve run external_scaler_config.yaml
```

In a separate terminal, run the predictive scaling client:

```bash
cd examples/custom-autoscaling-experiments/external_scaler/
python external_scaler_predictive_client.py
```

## Summary

| Policy Type | Complexity | Use Case | Key Feature |
|-------------|------------|----------|-------------|
| **Schedule-based** | Simple | Predictable traffic | Time-of-day scaling |
| **Recorded stats** | Medium | Resource-aware | CPU/memory from `record_autoscaling_stats()` |
| **Prometheus-based** | Advanced | Latency SLA | P90 latency from external monitoring |
| **Application-level** | Advanced | Pipeline coordination | Multi-deployment joint decisions |
| **External autoscaler** | Medium | Custom integrations | REST API for external control |

<div class="alert alert-block alert-info">

**Note:** Custom autoscaling is in alpha. Read more in [the Ray Serve docs](https://docs.ray.io/en/latest/serve/advanced-guides/advanced-autoscaling.html#custom-autoscaling-policies).

</div>

In [ ]:
# Cleanup
serve.shutdown()